# TripMe - Regenerate Place Descriptions (~100 words, LLM)

Reads every place record already collected under `data/raw/<District>/<category>.jsonl`,
rewrites each `description` field to a fresh ~100 word LLM-generated description
(Qwen2.5-3B-Instruct), and writes the updated records back out in the same
per-district/per-category JSONL layout - ready to copy back into `data/raw` locally.

## How to run this on Kaggle
1. Zip your local `data/raw` folder and upload it as a Kaggle dataset (e.g. named
   `tripme-raw-places`), then attach it to this notebook via "Add Input".
2. Set `RAW_INPUT_DIR` below to wherever Kaggle mounts it
   (usually `/kaggle/input/<dataset-name>/raw` or `/kaggle/input/<dataset-name>`
   depending on how the zip was structured - check the Kaggle "Input" file browser).
3. Turn on a GPU accelerator (Settings -> Accelerator -> GPU T4 x2 or better).
4. Run all cells. Output files land in `/kaggle/working/raw_updated/` - download
   that folder (Kaggle auto-zips the working directory) and copy its contents
   over `data/raw` locally.

This notebook is resumable: progress is checkpointed to
`/kaggle/working/desc_progress.json`, so if the Kaggle session gets interrupted
partway through, re-running continues where it left off instead of starting over.

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes sentencepiece tqdm
print("Dependencies installed.")

In [ ]:
import json
import re
from pathlib import Path

from tqdm.auto import tqdm

# Where the uploaded data/raw folder is mounted. Kaggle dataset slugs don't
# always match what you typed (e.g. "tripme-raw-places" vs "tripme-raw-place")
# and the folder structure inside the zip can vary, so search for it instead
# of hardcoding one exact path - much less fragile than a single guess.
RAW_INPUT_DIR = None
_kaggle_input = Path("/kaggle/input")
if _kaggle_input.exists():
    _candidates = [p for p in _kaggle_input.rglob("*") if p.is_dir() and p.name == "raw"]
    if not _candidates:
        # No "raw" subfolder found - maybe the zip was uploaded flat, so the
        # jsonl files sit directly under the dataset folder instead of a
        # nested "raw/" directory. Accept any dataset folder that directly
        # contains .jsonl files as a fallback.
        _candidates = [
            p for p in _kaggle_input.iterdir()
            if p.is_dir() and any(p.rglob("*.jsonl"))
        ]
    if _candidates:
        RAW_INPUT_DIR = _candidates[0]
        print(f"Auto-detected input dataset folder: {RAW_INPUT_DIR}")
    else:
        print("WARNING: /kaggle/input exists but no folder named 'raw' (or "
              "containing .jsonl files) was found inside it. Check the Input "
              "panel on the right and set RAW_INPUT_DIR manually below.")

if RAW_INPUT_DIR is None:
    # Fallback for local/non-Kaggle runs (e.g. testing on your own machine).
    RAW_INPUT_DIR = Path("data/raw")

OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
RAW_OUTPUT_DIR = OUTPUT_DIR / "raw_updated"
PROGRESS_FILE = OUTPUT_DIR / "desc_progress.json"

RAW_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Reading from:", RAW_INPUT_DIR)
print("Writing to:", RAW_OUTPUT_DIR)

## Load every place record from data/raw

In [ ]:
def load_records(path: Path):
    text = path.read_text(encoding="utf-8")
    decoder = json.JSONDecoder()
    idx = 0
    n = len(text)
    while idx < n:
        while idx < n and text[idx] in " \t\r\n":
            idx += 1
        if idx >= n:
            break
        obj, end = decoder.raw_decode(text, idx)
        yield obj
        idx = end


all_files = sorted(RAW_INPUT_DIR.rglob("*.jsonl"))
print(f"Found {len(all_files)} .jsonl files under {RAW_INPUT_DIR}")

records_by_file = {}
total_records = 0
for f in all_files:
    recs = list(load_records(f))
    records_by_file[f] = recs
    total_records += len(recs)

print(f"Loaded {total_records} place records total.")

## Load the LLM (Qwen2.5-3B-Instruct, 4-bit)

In [ ]:
LLM_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

llm_ready = False
llm_tokenizer = None
llm_model = None

try:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    if torch.cuda.is_available():
        print(f"Loading {LLM_MODEL_NAME} (4-bit)...")
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
        )
        llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)
        llm_model = AutoModelForCausalLM.from_pretrained(
            LLM_MODEL_NAME,
            quantization_config=bnb_config,
            device_map="auto",
        )
        llm_model.eval()
        llm_ready = True
        print("LLM loaded and ready.")
    else:
        print("No GPU detected. This notebook needs a GPU to generate real "
              "~100 word descriptions - enable one via Settings -> Accelerator "
              "and re-run. Continuing would only produce short template text, "
              "which defeats the point of this run, so stopping here.")
        raise SystemExit("No GPU available.")
except ImportError as e:
    raise SystemExit(f"Missing dependency ({e}) - re-run the pip install cell above.")

## Description generation (~100 words, grounded only in known facts)

In [ ]:
CATEGORY_BLURB = {
    "Buddhist Temple": "a Buddhist temple", "Kovil": "a Hindu kovil",
    "Church": "a church", "Mosque": "a mosque", "Temple": "a place of worship",
    "Ruins": "an archaeological site", "Building": "a historic landmark",
    "Cascade": "a waterfall", "Plunge": "a waterfall", "Tiered": "a waterfall",
    "Fan": "a waterfall", "Horsetail": "a waterfall", "Block": "a waterfall",
    "Segmented": "a waterfall", "Multi-step": "a waterfall",
    "Sandy Beach": "a beach", "Surf Beach": "a beach", "Urban Beach": "a beach",
    "Cove Beach": "a beach", "National Park": "a nature reserve",
    "Viewpoint": "a scenic viewpoint", "Museum": "a museum",
    "Adventure Park": "an adventure/activity park", "Tea Estate": "a tea estate",
    "Devalaya": "a Hindu-Buddhist shrine (devalaya)", "Devale": "a Hindu-Buddhist shrine (devale)",
    "Stupa": "a Buddhist stupa", "Fort": "a historic fort",
    "Other": "a point of interest",
}


def blurb_for(category_id: str) -> str:
    return CATEGORY_BLURB.get(category_id, "a point of interest")


DESC_SYSTEM_PROMPT = (
    "You write descriptions of Sri Lankan tourist places for a travel app. "
    "If you have genuine knowledge about this specific place - its history, "
    "who built it, when, its cultural or religious significance, notable "
    "events connected to it - include that, since it adds real value for "
    "travelers. But do NOT guess or invent history, dates, founders, or "
    "other specific facts you are not actually confident about: a wrong "
    "historical claim is worse than no historical claim. If you do not "
    "recognize this specific place, write a grounded description using only "
    "the name, category, district, and activities given to you, without "
    "inventing any history. Write exactly one paragraph, approximately 100 "
    "words long (90 to 110 words). Do not use markdown, headings, or bullet "
    "points. Reply with only "
    "the description text, no preamble, no word count, no disclaimers about "
    "uncertainty."
)


def build_desc_prompt(rec: dict) -> str:
    name = rec.get("name", "")
    category_id = rec.get("category_id", "Other")
    district = rec.get("district_id", "")
    activities = rec.get("activities", "")
    blurb = blurb_for(category_id)
    lines = [
        f'Place name: "{name}"',
        f"Category: {category_id} ({blurb})",
        f"District: {district}, Sri Lanka",
    ]
    if activities:
        lines.append(f"Known activities: {activities}")
    lines.append(
        "\nIf you genuinely know this place's history or significance, include "
        "it. Otherwise write a grounded description from the facts above only. "
        "Approximately 100 words (90-110)."
    )
    return "\n".join(lines)


# Batched generation: build prompts for a whole batch of records at once,
# left-pad and tokenize together, and run a single model.generate() call for
# the batch instead of one call per record. This is what actually keeps the
# GPU busy - a T4 sitting on one-record-at-a-time generate() calls is mostly
# idle waiting on Python/tokenizer overhead between tiny generations. Batches
# of 16-32 give a large (4-8x) throughput improvement over sequential calls.
BATCH_SIZE = 24

llm_tokenizer.padding_side = "left"
if llm_tokenizer.pad_token is None:
    llm_tokenizer.pad_token = llm_tokenizer.eos_token


def llm_generate_batch(recs: list[dict], max_new_tokens=170, temperature=0.6) -> list[str]:
    prompts = []
    for rec in recs:
        messages = [
            {"role": "system", "content": DESC_SYSTEM_PROMPT},
            {"role": "user", "content": build_desc_prompt(rec)},
        ]
        prompts.append(llm_tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=False
        ))
    inputs = llm_tokenizer(prompts, return_tensors="pt", padding=True).to(llm_model.device)
    with torch.no_grad():
        out = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=llm_tokenizer.eos_token_id,
        )
    input_len = inputs["input_ids"].shape[1]
    texts = llm_tokenizer.batch_decode(out[:, input_len:], skip_special_tokens=True)
    return [t.strip() for t in texts]


def is_valid_description(text: str, name: str) -> bool:
    if not text:
        return False
    word_count = len(text.split())
    if word_count < 80 or word_count > 130:
        return False
    first_word = re.sub(r"[^a-zA-Z]", "", name.split()[0]) if name.split() else ""
    if first_word and len(first_word) > 2 and first_word.lower() not in text.lower():
        return False
    return True


def generate_descriptions_batch(recs: list[dict], max_retries=3) -> dict:
    """Returns {id: description} for every record that produced a valid
    description within max_retries batched attempts. Records that still fail
    after all retries are simply absent from the result (caller keeps their
    old description as a fallback)."""
    results = {}
    remaining = list(recs)
    for attempt in range(max_retries + 1):
        if not remaining:
            break
        try:
            texts = llm_generate_batch(remaining)
        except Exception as e:
            print(f"  batch generation error (attempt {attempt+1}): {e}")
            continue
        still_remaining = []
        for rec, text in zip(remaining, texts):
            if is_valid_description(text, rec.get("name", "")):
                results[rec["id"]] = text
            else:
                still_remaining.append(rec)
        remaining = still_remaining
    return results


# Quick smoke test
sample_batch = [
    {"id": "test1", "name": "Aluvihara Rock Temple", "category_id": "Buddhist Temple",
     "district_id": "Matale", "activities": "meditation, photography"},
    {"id": "test2", "name": "Nine Arch Bridge", "category_id": "Building",
     "district_id": "Badulla", "activities": "photography, train watching"},
]
print(generate_descriptions_batch(sample_batch))

## Process every record (resumable via a checkpoint keyed by place id)

In [ ]:
import time

# Kaggle sessions get killed at a hard 12-hour wall-clock limit. Stop working
# (and save) well before that so we always exit cleanly with progress saved,
# rather than getting killed mid-generation. Re-running this notebook (or
# just re-running this cell) picks up exactly where it left off via the
# checkpoint file - nothing is lost by stopping early.
MAX_RUNTIME_HOURS = 9.0
run_deadline = time.time() + MAX_RUNTIME_HOURS * 3600


def load_progress() -> dict:
    if PROGRESS_FILE.exists():
        return json.loads(PROGRESS_FILE.read_text(encoding="utf-8"))
    return {"done": {}}  # id -> new description text


def save_progress(progress: dict) -> None:
    PROGRESS_FILE.write_text(json.dumps(progress), encoding="utf-8")


# Skip records whose existing description is already >= this many words.
# Plenty of records (the hand-curated ones) already have rich 100-300+ word
# descriptions - regenerating those with the LLM would be pure wasted GPU
# time for no quality gain, so only the short/template-generated ones
# (typically 20-49 words, well under the ~100-word target) actually need a
# new description.
MIN_WORDS_TO_SKIP = 90

progress = load_progress()
done_map = progress["done"]
print(f"Resuming: {len(done_map)} descriptions already generated in a previous run.")

all_records = [rec for recs in records_by_file.values() for rec in recs]
already_long_enough = sum(
    1 for rec in all_records
    if rec.get("id") not in done_map
    and len((rec.get("description") or "").split()) >= MIN_WORDS_TO_SKIP
)
pending = [
    rec for rec in all_records
    if rec.get("id") not in done_map
    and len((rec.get("description") or "").split()) < MIN_WORDS_TO_SKIP
]
print(f"{already_long_enough} records already have a >= {MIN_WORDS_TO_SKIP}-word description - skipping those.")
print(f"{len(pending)} of {len(all_records)} records need a new description.")
print(f"Batch size: {BATCH_SIZE}. Will stop generating (and save progress) after "
      f"{MAX_RUNTIME_HOURS} hours even if not finished - just re-run this notebook to continue.")

batches = [pending[i:i + BATCH_SIZE] for i in range(0, len(pending), BATCH_SIZE)]
stopped_early = False
for batch in tqdm(batches, desc="Generating descriptions (batched)"):
    if time.time() >= run_deadline:
        stopped_early = True
        print(f"\nHit the {MAX_RUNTIME_HOURS}-hour safety limit - stopping and saving progress.")
        break
    batch_results = generate_descriptions_batch(batch)
    done_map.update(batch_results)
    # else: records not in batch_results keep their old description as a
    # fallback when writing output below, and get retried on the next run.
    save_progress(progress)

save_progress(progress)
print(f"\nDone. {len(done_map)} descriptions generated/updated so far "
      f"({len(all_records) - len(done_map)} still pending"
      + (" - stopped early on the time limit, re-run to continue)." if stopped_early
         else " - re-run this cell to retry them)."))

## Write updated records back out, same per-district/per-category layout

In [ ]:
updated_count = 0
kept_old_count = 0

for src_path, recs in records_by_file.items():
    rel = src_path.relative_to(RAW_INPUT_DIR)
    out_path = RAW_OUTPUT_DIR / rel
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        for rec in recs:
            rid = rec.get("id")
            if rid in done_map:
                rec = dict(rec)
                rec["description"] = done_map[rid]
                updated_count += 1
            else:
                kept_old_count += 1
            f.write(json.dumps(rec, indent=4, ensure_ascii=False))
            f.write("\n\n")

print(f"Wrote {len(records_by_file)} files to {RAW_OUTPUT_DIR}")
print(f"{updated_count} records got a new ~100 word description.")
print(f"{kept_old_count} records kept their old description (generation failed every "
      f"retry, or run was interrupted before reaching them - re-run to fill these in).")
print("\nNext step: download /kaggle/working (Kaggle auto-zips it), then copy the "
      "contents of raw_updated/ over your local data/raw folder.")

In [ ]:
import shutil

# Kaggle has no direct browser-auto-download API (unlike Colab's
# files.download()) - the only way to get files out is: zip them here,
# click "Save Version" (top right) so Kaggle persists /kaggle/working as a
# Version Output, then download the zip from the notebook's "Output" tab.
zip_path = shutil.make_archive("/kaggle/working/raw_updated", "zip", RAW_OUTPUT_DIR)
print(f"Zipped to {zip_path}")
print("\nNext: click 'Save Version' (top right) to persist this Output, then "
      "once it finishes, open the 'Output' tab on the right and download "
      "raw_updated.zip - that's what you copy over your local data/raw folder.")